# Neural ODE Influenza Forecaster -- Part 3: Competition Workflow

This notebook is the third of three. It contains only what you need on competition
day: a self-contained `CompetitionSession` class and one live cell per round.

**Three-notebook structure:**
- **Part 1:** Data
- **Part 2:** Model
- **Part 3 (this notebook):** Competition workflow -- the live submission interface

## How to use this notebook

1. At the start of the competition season, run all cells top to bottom once.
   This defines `CompetitionSession` and creates `live_session`.

2. Each time a batch of data is released:
   - Paste the new `new_weeks` and `new_H` arrays into the live cell at the bottom.
   - Run that cell.
   - The submission table is printed automatically. Copy the values into your submission.
   - Call `live_session.plot_current()` to visualize the forecast.

3. Do not restart the kernel between rounds. The session accumulates history
   across calls; restarting would lose the history and require re-pasting all
   prior data.

## What this notebook does not explain

The architecture, training loop, ensemble construction, and R(t) estimation are
all documented in Part 2. This notebook treats those as a black box and focuses
entirely on the submission interface. If a forecast looks wrong, the place to
debug is Part 2.

## 1. Setup

In [ ]:
# torchdiffeq provides the ODE solvers (Dopri5 / RK45 by default) and the adjoint method
!pip install torchdiffeq --quiet

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import copy
import io
from torchdiffeq import odeint_adjoint as odeint

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

Using device: cpu
PyTorch version: 2.11.0+cpu


In [ ]:
DATA_DIR = '.'   # directory containing your competition CSVs

def load_season(filename):
    df = pd.read_csv(filename, encoding='utf-8-sig')
    cols = df.columns.tolist()
    df = df.rename(columns={cols[0]: 'week', cols[1]: 'hospitalizations'})
    df['week']            = df['week'].astype(int)
    df['hospitalizations'] = df['hospitalizations'].astype(float)
    return df

def normalize_season(df):
    H     = df['hospitalizations'].values.astype(np.float32)
    scale = float(H.max())
    return H / scale, scale

print('Data utilities defined.')

## 2. Model Definitions

All model code reproduced here so this notebook is self-contained.

In [24]:
class ODEFunc(nn.Module):
    """
    f_theta: the right-hand side of the neural ODE.

    Input:  (t, x) where t is a scalar and x has shape (batch, latent_dim)
    Output: dx/dt of shape (batch, latent_dim)

    torchdiffeq calls this as func(t, x) repeatedly during integration.
    Note: torchdiffeq passes t as a 0-dim tensor, not batched.
    """

    def __init__(self, latent_dim=6, hidden_dim=64, include_time=True):
        super().__init__()
        self.include_time = include_time
        in_dim = latent_dim + (1 if include_time else 0)

        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.Tanh(),                      # Tanh preferred over ReLU for ODEs:
            nn.Linear(hidden_dim, hidden_dim),  # smoother derivatives, better
            nn.Tanh(),                          # behaved gradients through the solver
            nn.Linear(hidden_dim, latent_dim),
        )

        # Small initialization: we want the ODE to start near x'=0
        # so training begins from a slowly-changing baseline
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.1)
                nn.init.zeros_(m.bias)

    def forward(self, t, x):
        if self.include_time:
            # t is a scalar (0-dim tensor); expand to (batch, 1)
            t_batch = t.expand(x.shape[0], 1)
            xt = torch.cat([x, t_batch], dim=1)
        else:
            xt = x
        return self.net(xt)


class NeuralODEForecaster(nn.Module):
    """
    Full model: initial condition encoder + ODE dynamics + observation decoder.

    Initial condition: a learnable parameter x0, shared across training sequences.
    For a more powerful model you would use an RNN encoder to infer x0 from
    the first few observed points (see Section 6 extension).

    Decoder: a single linear layer + softplus maps latent -> hospitalization rate.
    """

    def __init__(self, latent_dim=6, hidden_dim=64, include_time=True):
        super().__init__()
        self.latent_dim = latent_dim

        self.ode_func = ODEFunc(latent_dim, hidden_dim, include_time)

        # Learnable initial condition in latent space
        self.x0 = nn.Parameter(torch.randn(1, latent_dim) * 0.1)

        # Decoder: latent -> observed H(t)
        self.decoder = nn.Linear(latent_dim, 1)

    def forward(self, t_span):
        """
        Integrate the ODE from t_span[0] to t_span[-1].

        Args:
            t_span: 1-D tensor of times at which to evaluate the solution
                    (must be sorted ascending)

        Returns:
            H_pred: shape (len(t_span),) -- predicted normalized hospitalizations
        """
        # odeint signature: odeint(func, y0, t)
        # y0 shape: (batch, latent_dim) -- here batch=1
        x_traj = odeint(
            self.ode_func,
            self.x0,
            t_span,
            method='dopri5',        # adaptive Runge-Kutta 4/5
            rtol=1e-3,              # looser tolerance = faster, fine for training
            atol=1e-4,
        )  # shape: (len(t_span), 1, latent_dim)

        x_traj = x_traj.squeeze(1)  # (len(t_span), latent_dim)

        # Decode to observation space
        # softplus ensures non-negative output
        H_pred = torch.nn.functional.softplus(self.decoder(x_traj)).squeeze(-1)
        return H_pred, x_traj


# Quick sanity check: make sure shapes are right before training
model_test = NeuralODEForecaster().to(DEVICE)
t_test = torch.linspace(0, 28, 29).to(DEVICE)
H_out, x_out = model_test(t_test)
print(f'Output H shape:      {H_out.shape}   (expected: torch.Size([29]))')
print(f'Latent traj shape:   {x_out.shape}   (expected: torch.Size([29, 6]))')
del model_test

Output H shape:      torch.Size([29])   (expected: torch.Size([29]))
Latent traj shape:   torch.Size([29, 6])   (expected: torch.Size([29, 6]))


In [25]:
def train_model(
    H_obs,           # normalized hospitalization array, shape (n_weeks,)
    weeks_obs,       # corresponding week integers
    n_epochs=300,
    lr=5e-3,
    latent_dim=6,
    hidden_dim=64,
    verbose=True,
    seed=SEED,
):
    """
    Train a NeuralODEForecaster on the provided observations.

    Returns:
        model: trained NeuralODEForecaster (best validation checkpoint)
        losses: list of per-epoch training losses
    """
    torch.manual_seed(seed)

    model = NeuralODEForecaster(latent_dim=latent_dim, hidden_dim=hidden_dim).to(DEVICE)

    # Tensors on device
    t_obs = torch.tensor(weeks_obs, dtype=torch.float32).to(DEVICE)
    H_target = torch.tensor(H_obs, dtype=torch.float32).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=1e-5)

    losses = []
    best_loss = float('inf')
    best_state = None

    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()

        H_pred, _ = model(t_obs)

        loss = nn.MSELoss()(H_pred, H_target)
        loss.backward()

        # Gradient clipping: Neural ODEs can have large gradients early in training
        # if the ODE is stiff. Clipping prevents divergence.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        l = loss.item()
        losses.append(l)

        if l < best_loss:
            best_loss = l
            best_state = copy.deepcopy(model.state_dict())

        if verbose and (epoch % 50 == 0 or epoch == n_epochs - 1):
            print(f'  Epoch {epoch:4d} | Loss: {l:.6f} | LR: {scheduler.get_last_lr()[0]:.2e}')

    # Restore best checkpoint
    model.load_state_dict(best_state)
    model.eval()
    return model, losses


print('Training function defined.')

Training function defined.


In [ ]:
N_ENSEMBLE = 5

def train_ensemble(H_obs, weeks_obs, n_models=N_ENSEMBLE, n_epochs=400, scale=1.0):
    ensemble = []
    for m in range(n_models):
        print(f'  model {m+1}/{n_models}...', end=' ', flush=True)
        model, losses = train_model(
            H_obs=H_obs, weeks_obs=weeks_obs,
            n_epochs=n_epochs, seed=SEED + m * 17,
            verbose=False,
        )
        print(f'loss {losses[-1]:.5f}')
        ensemble.append(model)
    return ensemble


def ensemble_forecast(ensemble, t_full_span, cutoff, scale=1.0):
    all_preds = []
    for model in ensemble:
        with torch.no_grad():
            H_span, _ = model(t_full_span)
        H_forecast = H_span[cutoff + 1:].cpu().numpy() * scale
        all_preds.append(H_forecast)
    all_preds  = np.array(all_preds)
    fore_weeks = np.arange(cutoff + 1, cutoff + 1 + all_preds.shape[1])
    return {
        'weeks':    fore_weeks,
        'median':   np.percentile(all_preds, 50,   axis=0),
        'q25':      np.percentile(all_preds, 25,   axis=0),
        'q75':      np.percentile(all_preds, 75,   axis=0),
        'q025':     np.percentile(all_preds,  2.5, axis=0),
        'q975':     np.percentile(all_preds, 97.5, axis=0),
        'all_preds': all_preds,
    }


print('Ensemble functions defined.')

In [ ]:
from scipy.stats import gamma as gamma_dist

def serial_interval_weights(max_tau=6, mean_si=2.0, std_si=1.0):
    shape = (mean_si / std_si) ** 2
    scale = (std_si ** 2) / mean_si
    taus  = np.arange(1, max_tau + 1, dtype=np.float64)
    w     = gamma_dist.pdf(taus, a=shape, scale=scale)
    return w / w.sum()


def estimate_rt_from_model(model, t_max=30, mean_si=2.0, std_si=1.0):
    t_dense = torch.linspace(0, t_max, t_max + 1).to(DEVICE)
    with torch.no_grad():
        H_pred, _ = model(t_dense)
    H = H_pred.cpu().numpy().astype(np.float64)
    H = np.clip(H, 1e-8, None)

    w       = serial_interval_weights(mean_si=mean_si, std_si=std_si)
    max_tau = len(w)
    rt_vals = np.full(len(H), np.nan)
    for t in range(max_tau, len(H)):
        denom   = sum(w[tau - 1] * H[t - tau] for tau in range(1, max_tau + 1))
        rt_vals[t] = H[t] / denom if denom > 1e-10 else np.nan
    return np.arange(len(H), dtype=np.float32), rt_vals


print('R(t) estimation function defined.')

In [ ]:
# ── Competition hyperparameters ───────────────────────────────────────────────
# Tune these on the sample data before competition day.

COMP_CONFIG = dict(
    first_batch_size   = 10,      # weeks in round-0 release
    subsequent_size    = 5,      # weeks in each subsequent release
    predict_horizon    = 5,      # weeks to forecast (one full subsequent batch)
    n_epochs           = 400,    # training epochs per round
    lr                 = 5e-3,   # initial learning rate
    latent_dim         = 6,      # latent state dimension
    hidden_dim         = 64,     # neurons per hidden layer
    n_ensemble         = 5,      # models per ensemble (for uncertainty intervals)
    retrain_from_scratch = True, # False = fine-tune from previous round's weights
)

print('Competition config:')
for k, v in COMP_CONFIG.items():
    print(f'  {k:<26} {v}')

In [44]:
class CompetitionSession:
    """
    Stateful object that manages one competition season.

    Usage (live competition):
        session = CompetitionSession(COMP_CONFIG)

        # Round 0: paste in the first 8 weeks from the release
        forecast = session.ingest(
            new_weeks = np.array([0,1,2,3,4,5,6,7]),
            new_H     = np.array([...]),   # actual released values
        )
        # forecast['table'] prints the submission-ready numbers

        # Round 1: paste in the 6 new weeks you just received
        forecast = session.ingest(
            new_weeks = np.array([8,9,10,11,12,13]),
            new_H     = np.array([...]),
        )
        # ... and so on each round
    """

    def __init__(self, config):
        self.cfg            = config
        self.history_weeks  = np.array([], dtype=np.float32)
        self.history_H      = np.array([], dtype=np.float32)
        self.round          = 0
        self.scale          = None   # set on first ingest; all H normalized by this
        self.last_ensemble  = None   # kept so you can inspect after each round
        self.results        = []     # log of every round's forecast

    # ------------------------------------------------------------------ #
    # Public interface                                                      #
    # ------------------------------------------------------------------ #

    def ingest(self, new_weeks, new_H):
        """
        Receive a new batch of real data, retrain, and return a forecast.

        Args:
            new_weeks : 1-D int/float array of week indices in this batch
            new_H     : 1-D float array of hospitalization values (per 100k),
                        same length as new_weeks

        Returns:
            dict with keys:
                round           int
                history_weeks   all weeks seen so far
                forecast_weeks  weeks being predicted
                median          median prediction (per 100k)
                q25, q75        50% credible interval
                q025, q975      95% credible interval
                rt_median       per-forecast-week R(t) estimates
                table           formatted string ready to copy-paste
        """
        new_weeks = np.asarray(new_weeks, dtype=np.float32)
        new_H     = np.asarray(new_H,     dtype=np.float32)

        assert len(new_weeks) == len(new_H), "weeks and H must be the same length"

        # ── 1. Append to history ─────────────────────────────────────────
        self.history_weeks = np.concatenate([self.history_weeks, new_weeks])
        self.history_H     = np.concatenate([self.history_H,     new_H])

        # ── 2. Set scale on first batch; reuse every subsequent round ────
        # Normalizing by the running max would shift the target values for
        # already-seen weeks each round, destabilizing training. Fix the
        # scale to the first batch maximum and never change it.
        if self.scale is None:
            self.scale = float(self.history_H.max())
            if self.scale < 1e-6:
                raise ValueError('First batch is all zeros -- cannot set scale')

        H_norm = self.history_H / self.scale

        # ── 3. Train ensemble ────────────────────────────────────────────
        print(f'\n=== Round {self.round} | '
              f'History: weeks {int(self.history_weeks[0])}-{int(self.history_weeks[-1])} '
              f'({len(self.history_weeks)} pts) ===')

        ensemble = self._train_ensemble(H_norm, self.history_weeks)
        self.last_ensemble = ensemble

        # ── 4. Forecast ──────────────────────────────────────────────────
        last_week      = int(self.history_weeks[-1])
        forecast_weeks = np.arange(
            last_week + 1,
            last_week + 1 + self.cfg['predict_horizon'],
            dtype=np.float32,
        )

        t_span = torch.arange(
            0, last_week + 1 + self.cfg['predict_horizon'],
            dtype=torch.float32,
        ).to(DEVICE)

        all_preds = []
        for model in ensemble:
            with torch.no_grad():
                H_full, _ = model(t_span)
            H_fore = H_full[last_week + 1:].cpu().numpy() * self.scale
            all_preds.append(H_fore)

        all_preds = np.array(all_preds)  # (n_ensemble, predict_horizon)

        # ── 5. Rt per ensemble member ────────────────────────────────────
        # R(t) is the time-varying effective reproduction number.
        # We compute it at each forecast week from the Neural ODE curve,
        # then take the median across ensemble members.
        rt_all = []
        for model in ensemble:
            _, rt_full = estimate_rt_from_model(
                model,
                t_max=last_week + self.cfg['predict_horizon'],
            )
            # Slice to only the forecast weeks
            rt_fore = rt_full[last_week + 1:]
            rt_all.append(rt_fore)

        rt_all    = np.array(rt_all)   # (n_ensemble, predict_horizon)
        rt_median = np.nanmedian(rt_all, axis=0)

        result = {
            'round':          self.round,
            'history_weeks':  self.history_weeks.copy(),
            'forecast_weeks': forecast_weeks,
            'median':  np.percentile(all_preds, 50,  axis=0),
            'q25':     np.percentile(all_preds, 25,  axis=0),
            'q75':     np.percentile(all_preds, 75,  axis=0),
            'q025':    np.percentile(all_preds,  2.5, axis=0),
            'q975':    np.percentile(all_preds, 97.5, axis=0),
            'all_preds': all_preds,
            'scale':     self.scale,
            'rt_median': rt_median,
        }
        result['table'] = self._format_table(result)

        self.results.append(result)
        self.round += 1

        print(result['table'])
        return result

    def plot_current(self, truth_weeks=None, truth_H=None):
        """
        Plot history, ensemble fit over history, and forecast with intervals.
        Optionally overlay ground truth in the forecast window if known.
        Call after ingest().
        """
        if not self.results:
            print('No rounds completed yet.')
            return

        res = self.results[-1]
        last_week = int(self.history_weeks[-1])

        fig, ax = plt.subplots(figsize=(11, 5))

        # ── History bars ─────────────────────────────────────────────────
        ax.bar(self.history_weeks, self.history_H,
               color='steelblue', alpha=0.75, label='Observed history')

        # ── Ensemble fit over history (median + 95% band) ─────────────────
        t_hist = torch.arange(0, last_week + 1, dtype=torch.float32).to(DEVICE)
        hist_preds = []
        for model in self.last_ensemble:
            with torch.no_grad():
                H_full, _ = model(t_hist)
            hist_preds.append(H_full.cpu().numpy() * self.scale)
        hist_preds = np.array(hist_preds)

        t_hist_np = t_hist.cpu().numpy()
        ax.plot(t_hist_np, np.median(hist_preds, axis=0),
                color='navy', linewidth=1.5, label='Ensemble fit')
        ax.fill_between(t_hist_np,
                        np.percentile(hist_preds, 2.5, axis=0),
                        np.percentile(hist_preds, 97.5, axis=0),
                        alpha=0.15, color='navy')

        # ── Forecast ─────────────────────────────────────────────────────
        fw = res['forecast_weeks']
        ax.fill_between(fw, res['q025'], res['q975'],
                        alpha=0.20, color='crimson', label='95% CI')
        ax.fill_between(fw, res['q25'], res['q75'],
                        alpha=0.40, color='crimson', label='50% CI')
        ax.plot(fw, res['median'], 'o-', color='crimson',
                linewidth=2, markersize=7, label='Forecast median')

        for pred in res['all_preds']:
            ax.plot(fw, pred, color='crimson', alpha=0.12, linewidth=1)

        # ── Optional ground truth overlay ─────────────────────────────────
        if truth_weeks is not None and truth_H is not None:
            ax.bar(truth_weeks, truth_H,
                   color='lightgray', alpha=0.9, label='True (revealed)')
            tw = np.asarray(truth_weeks)
            th = np.asarray(truth_H)
            match = np.isin(fw.astype(int), tw.astype(int))
            if match.any():
                pred_matched  = res['median'][match]
                truth_matched = th[np.isin(tw.astype(int), fw[match].astype(int))]
                rmse = np.sqrt(np.mean((pred_matched - truth_matched) ** 2))
                ax.set_title(
                    f'Round {res["round"]} | '
                    f'History weeks 0-{last_week} | '
                    f'Forecast RMSE: {rmse:.3f} per 100k'
                )
            else:
                ax.set_title(f'Round {res["round"]} | History weeks 0-{last_week}')
        else:
            ax.set_title(f'Round {res["round"]} | History weeks 0-{last_week}')

        ax.axvline(last_week + 0.5, color='black', linestyle=':', alpha=0.5,
                   label='Data cutoff')
        ax.set_xlabel('Week')
        ax.set_ylabel('Hospitalizations per 100k')
        ax.legend(fontsize=9)
        plt.tight_layout()
        plt.show()

    def plot_all_rounds(self, truth_weeks=None, truth_H=None):
        """
        Summary grid: one panel per completed round.
        Pass the full ground-truth arrays to score every round simultaneously.
        Only useful in simulation (run_on_sample) where you have the full truth.
        """
        n = len(self.results)
        if n == 0:
            print('No rounds completed yet.')
            return

        ncols = min(n, 3)
        nrows = (n + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols,
                                  figsize=(5.5 * ncols, 4.5 * nrows),
                                  sharey=True)
        axes = np.array(axes).flatten()

        for i, res in enumerate(self.results):
            ax = axes[i]
            hist_end = int(res['history_weeks'][-1])

            n_hist = len(res['history_weeks'])
            H_hist = self.history_H[:n_hist] if i == n - 1 else (
                np.concatenate([self.history_H[:n_hist]])
            )
            ax.bar(res['history_weeks'], H_hist,
                   color='steelblue', alpha=0.75)

            fw = res['forecast_weeks']
            ax.fill_between(fw, res['q025'], res['q975'],
                            alpha=0.2, color='crimson')
            ax.fill_between(fw, res['q25'], res['q75'],
                            alpha=0.4, color='crimson')
            ax.plot(fw, res['median'], 'o-', color='crimson',
                    linewidth=2, markersize=6)

            rmse_str = ''
            if truth_weeks is not None and truth_H is not None:
                tw = np.asarray(truth_weeks)
                th = np.asarray(truth_H)
                mask = np.isin(tw.astype(int), fw.astype(int))
                if mask.any():
                    ax.bar(tw[mask], th[mask], color='lightgray', alpha=0.9)
                    pred_m  = res['median'][np.isin(fw.astype(int), tw[mask].astype(int))]
                    truth_m = th[mask]
                    n_c = min(len(pred_m), len(truth_m))
                    if n_c:
                        rmse = np.sqrt(np.mean((pred_m[:n_c] - truth_m[:n_c]) ** 2))
                        rmse_str = f'  RMSE={rmse:.2f}'

            ax.axvline(hist_end + 0.5, color='black', linestyle=':', alpha=0.5)
            ax.set_title(f'Round {res["round"]} | weeks 0-{hist_end}{rmse_str}',
                         fontsize=10)
            ax.set_xlabel('Week')
            ax.set_ylabel('per 100k')

        for j in range(i + 1, len(axes)):
            axes[j].set_visible(False)

        plt.suptitle('All Rounds -- Neural ODE Competition Forecasts', fontsize=13)
        plt.tight_layout()
        plt.show()

    # ------------------------------------------------------------------ #
    # Private helpers                                                       #
    # ------------------------------------------------------------------ #

    def _train_ensemble(self, H_norm, weeks):
        n = self.cfg['n_ensemble']
        ensemble = []
        for m in range(n):
            print(f'  model {m+1}/{n}...', end=' ', flush=True)
            model, losses = train_model(
                H_obs      = H_norm,
                weeks_obs  = weeks,
                n_epochs   = self.cfg['n_epochs'],
                lr         = self.cfg['lr'],
                latent_dim = self.cfg['latent_dim'],
                hidden_dim = self.cfg['hidden_dim'],
                verbose    = False,
                seed       = SEED + m * 17 + self.round * 100,
            )
            print(f'loss {losses[-1]:.5f}')
            ensemble.append(model)
        return ensemble

    @staticmethod
    def _format_table(res):
        lines = [
            f'\nSubmission -- Round {res["round"]}',
            f'{"Week":>6}  {"Hospitalizations_per_100K":>26}  {"r0":>8}',
            '-' * 46,
        ]
        rt = res.get('rt_median', None)
        for i, w in enumerate(res['forecast_weeks']):
            rt_str = f'{rt[i]:>8.3f}' if (rt is not None and not np.isnan(rt[i])) else '     N/A'
            lines.append(
                f'{int(w):>6}  '
                f'{res["median"][i]:>26.3f}  '
                f'{rt_str}'
            )
        lines.append('-' * 46)
        return '\n'.join(lines)


print('CompetitionSession defined.')

CompetitionSession defined.


## 3. Live Competition Cell

Run this section each time a new data batch is released.
Replace `new_weeks` and `new_H` with the values from the release.
The cell appends to the session's history, retrains, and prints the submission table.

**Do not re-run the session creation line (`live_session = CompetitionSession(COMP_CONFIG)`)
between rounds.** That resets all history.

In [ ]:
# ── Create once per competition season ──────────────────────────────────────
# Run this line ONCE. It resets all history.
live_session = CompetitionSession(COMP_CONFIG)

In [ ]:
# ── Round 1: weeks 1-10 released ────────────────────────────────────────────
new_weeks = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=np.float32)
new_H     = np.array([0.1, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.3], dtype=np.float32)

forecast = live_session.ingest(new_weeks, new_H)
live_session.plot_current()

In [ ]:
# ── Round 2: weeks 11-15 released ───────────────────────────────────────────
new_weeks = np.array([11, 12, 13, 14, 15], dtype=np.float32)
new_H     = np.array([1.5, 1.8, 2.0, 2.1, 2.3], dtype=np.float32)

forecast = live_session.ingest(new_weeks, new_H)
live_session.plot_current()

In [ ]:
# ── Round 3: weeks 16-20 released ───────────────────────────────────────────
new_weeks = np.array([16, 17, 18, 19, 20], dtype=np.float32)
new_H     = np.array([2.6, 3.1, 3.7, 4.5, 5.2], dtype=np.float32)

forecast = live_session.ingest(new_weeks, new_H)
live_session.plot_current()

In [ ]:
# ── Round 4: weeks 21-25 released ───────────────────────────────────────────
new_weeks = np.array([21, 22, 23, 24, 25], dtype=np.float32)
new_H     = np.array([5.3, 4.8, 3.8, 2.8, 1.9], dtype=np.float32)

forecast = live_session.ingest(new_weeks, new_H)
live_session.plot_current()